In [1]:
import pandas as pd
import numpy as np
from calendar import monthrange
from sqlalchemy import create_engine, text

In [2]:
# def create_date_column(df):
#     # 각 월의 마지막 날짜 계산 함수
#     def get_last_day_of_month(year, month):
#         return monthrange(int(year), int(month))[1]
#
#     # 날짜 열 생성
#     df['Date'] = df.apply(
#         lambda row: pd.to_datetime(
#             f"{int(row['회계년'])}-{int(row['결산월'])}-{get_last_day_of_month(row['회계년'], row['결산월'])}"
#         ),
#         axis=1
#     )
#
#     # 주기가 분기형이면, 날짜를 분기 말일로 보정 (선택적 처리)
#     if '주기' in df.columns:
#         quarterly_mask = df['주기'].astype(str).str.contains("Q")
#         for idx in df[quarterly_mask].index:
#             base_date = df.loc[idx, 'Date']
#             df.loc[idx, 'Date'] = pd.date_range(end=base_date, periods=4, freq='3M')[-1]
#
#     return df

def convert_to_long_format(df):
    # 제거할 열
    drop_cols = ["결산월", "회계년", "주기"]

    # ID 변수 (고정값 유지할 열들)
    id_vars = ["Symbol", "company_name", "Date"]

    # 나머지는 전부 indicator 대상 열로 melt 처리
    value_vars = [col for col in df.columns if col not in id_vars + drop_cols]

    # melt 실행
    df_long = pd.melt(df,
                      id_vars=id_vars,
                      value_vars=value_vars,
                      var_name="indicator",
                      value_name="value")

    return df_long


def create_date_column(df):
    # 각 월의 마지막 날짜 계산 함수
    def get_last_day_of_month(year, month):
        return monthrange(int(year), int(month))[1]

    # 날짜 열 생성
    df['Date'] = df.apply(
        lambda row: pd.to_datetime(
            f"{int(row['회계년'])}-{int(row['결산월'])}-{get_last_day_of_month(row['회계년'], row['결산월'])}"
        ),
        axis=1
    )

    # 주기가 분기형이면, 날짜를 해당 분기 말일로 직접 지정
    if '주기' in df.columns:
        quarter_map = {
            '1Q': '-03-31',
            '2Q': '-06-30',
            '3Q': '-09-30',
            '4Q': '-12-31'
        }

        def override_to_quarter_end(row):
            q = str(row['주기']).strip()
            y = int(row['회계년'])
            return pd.to_datetime(f"{y}{quarter_map[q]}") if q in quarter_map else row['Date']

        df['Date'] = df.apply(override_to_quarter_end, axis=1)

    return df

def upload_fs_data_to_db(df_long, db_info, table_name="korea_fs_data", chunk_size=1000):
    """
    DB에 long format 재무 데이터를 테이블 생성 후 업로드함.
    기존 테이블은 없다고 가정하고 새로 생성함.
    """
    # ✅ 날짜 및 결측치 처리
    df_long['date'] = pd.to_datetime(df_long['date'])
    df_long = df_long.replace([np.inf, -np.inf], np.nan)
    df_long = df_long.where(pd.notnull(df_long), None)

    # ✅ DB 연결
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    conn = engine.raw_connection()
    cursor = conn.cursor()

    # ✅ 테이블 생성 쿼리
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
        `symbol` VARCHAR(20),
        `company_name` VARCHAR(50),
        `date` DATE,
        `indicator` VARCHAR(100),
        `value` DOUBLE,
        PRIMARY KEY (`symbol`, `date`, `indicator`)
    );
    """
    cursor.execute(create_table_sql)
    conn.commit()

    # ✅ 데이터 INSERT 쿼리
    insert_sql = f"""
    INSERT INTO `{table_name}` (`symbol`, `company_name`, `date`, `indicator`, `value`)
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        `company_name` = VALUES(`company_name`),
        `value` = VALUES(`value`);
    """

    # ✅ 튜플 리스트로 변환
    rows = df_long[["symbol", "company_name", "date", "indicator", "value"]].values.tolist()

    # ✅ Chunk 단위로 업로드
    for i in range(0, len(rows), chunk_size):
        chunk = rows[i:i+chunk_size]
        cursor.executemany(insert_sql, chunk)
        conn.commit()

    cursor.close()
    conn.close()
    print(f"✅ 총 {len(df_long)}개 row 업로드 완료 (중복은 자동 업데이트됨)")


    def convert_to_long_format(df):
        # 제거할 열
        drop_cols = ["결산월", "회계년", "주기"]

        # ID 변수 (고정값 유지할 열들)
        id_vars = ["Symbol", "company_name", "Date"]

        # 나머지는 전부 indicator 대상 열로 melt 처리
        value_vars = [col for col in df.columns if col not in id_vars + drop_cols]

        # melt 실행
        df_long = pd.melt(df,
                          id_vars=id_vars,
                          value_vars=value_vars,
                          var_name="indicator",
                          value_name="value")

        return df_long

In [3]:
path = r"C:\Users\MetaM\PycharmProjects\stock_forecast\DATA\KSE_FS_Dataguide.xlsx"

raw_df = pd.read_excel(path)

# 기준 행
row_header_1 = 8  # 항목명
row_header_2 = 9  # 단위, 코드, 분류 등

# 복합 컬럼명 생성
def combine_headers(col):
    item = str(raw_df.loc[row_header_1, col]) if pd.notna(raw_df.loc[row_header_1, col]) else ""
    unit = str(raw_df.loc[row_header_2, col]) if pd.notna(raw_df.loc[row_header_2, col]) else ""
    combined = f"{unit.strip()}: {item.strip()}" if unit and item else item or unit
    return combined if combined else col  # fallback

# 새로운 컬럼 리스트 생성
new_columns = [combine_headers(col) for col in raw_df.columns]

# 컬럼명 적용
df_cleaned = raw_df.copy()
df_cleaned.columns = new_columns
df_cleaned = df_cleaned.iloc[10:].reset_index(drop=True)

# "Name"을 "company_name"으로 변경
df_cleaned = df_cleaned.rename(columns={"Name": "company_name"})

# 확인
print(df_cleaned.columns.tolist())

['Symbol', 'company_name', '결산월', '회계년', '주기', '매출액(천원)', '매출총이익(천원)', '영업이익(천원)', '계속사업이익(천원)', '당기순이익(천원)', '지배주주총포괄이익(천원)', '유형자산감가상각비(천원)', '연구개발비(천원)', '총자산(천원)', '유동자산(천원)', '당좌자산(천원)', '매출채권및기타채권(천원)', '재고자산(천원)', '투자부동산(천원)', '비유동부채(천원)', '총자본(천원)', '지배주주지분(천원)', '영업활동으로인한현금흐름(천원)', '영업활동으로인한현금흐름(직전4분기)(천원)', '영업활동으로인한현금흐름(평균)(천원)', '배당금지급(영업,투자,재무)(천원)']


In [4]:
# ======================
# 1. 주요 지표 컬럼 이름 찾기
# ======================
def find_column(df, keyword):
    for col in df.columns:
        if keyword in col and '(천원)' in col:
            return col
    return None

# 주요 항목들 컬럼명 식별
col_sales = find_column(df_cleaned, '매출액')
col_gross = find_column(df_cleaned, '매출총이익')
col_operating = find_column(df_cleaned, '영업이익')
col_continuing = find_column(df_cleaned, '계속사업이익')
col_net_income = find_column(df_cleaned, '당기순이익')
col_noncurrent_liab = find_column(df_cleaned, '비유동부채')
col_equity = None
for col in df_cleaned.columns:
    if "지배" in col and "지분" in col:
        col_equity = col
        break

# ======================
# 2. 숫자형으로 변환
# ======================
for col in [col_sales, col_gross, col_operating, col_continuing, col_net_income, col_noncurrent_liab, col_equity]:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# ======================
# 3. sort와 groupby 이용하여 YoY 증가율 계산
# ======================
df_cleaned = df_cleaned.sort_values(by=["Symbol", "회계년", "주기"])

# 증가율 계산 함수
def calc_yoy(df, col):
    return df[col].pct_change(periods=4)

# YoY 증가율 계산 (4분기 전과 비교)
df_cleaned["YoY_매출액"] = df_cleaned.groupby("Symbol")[col_sales].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_매출총이익"] = df_cleaned.groupby("Symbol")[col_gross].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_영업이익"] = df_cleaned.groupby("Symbol")[col_operating].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_계속사업이익"] = df_cleaned.groupby("Symbol")[col_continuing].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_당기순이익"] = df_cleaned.groupby("Symbol")[col_net_income].pct_change(periods=4, fill_method=None)

# ======================
# 4. 수익성 비율 계산
# ======================
# 수익성 비율
df_cleaned["매출총이익률"] = df_cleaned[col_gross] / df_cleaned[col_sales]
df_cleaned["영업이익률"] = df_cleaned[col_operating] / df_cleaned[col_sales]
df_cleaned["순이익률"] = df_cleaned[col_net_income] / df_cleaned[col_sales]

# 재무 안정성 비율
df_cleaned["비유동부채비율"] = df_cleaned[col_noncurrent_liab] / df_cleaned[col_equity]

# ======================
# 결과 미리보기
# ======================
display_cols = [
    "Symbol", "회계년", "주기", col_sales,
    "YoY_매출액", "YoY_영업이익", "영업이익률", "순이익률", "비유동부채비율"
]
print(df_cleaned[display_cols].dropna().head(10))

        Symbol   회계년  주기      매출액(천원)   YoY_매출액  YoY_영업이익     영업이익률      순이익률  \
14788  A000080  2005  1Q  178510186.0  0.119081  0.534541  0.328599  0.248931   
14789  A000080  2005  2Q  178139523.0  0.090801  0.311304  0.312639  0.994690   
14790  A000080  2005  3Q  185444855.0 -0.006677 -0.385215  0.206434  1.251166   
14791  A000080  2005  4Q  187615442.0  0.022223  0.143054  0.308197  0.312691   
14792  A000080  2006  1Q  180254189.0  0.009770 -0.371722  0.204454  0.143913   
14793  A000080  2006  2Q  174980431.0 -0.017734 -0.356911  0.204685  0.398813   
14794  A000080  2006  3Q  165448485.0 -0.107829 -0.365278  0.146865  0.130170   
14795  A000080  2006  4Q  180111181.0 -0.039998 -0.551457  0.143999 -0.007284   
14796  A000080  2007  1Q  165170484.0 -0.083680 -0.112984  0.197916  0.158397   
14797  A000080  2007  2Q  158097052.0 -0.096487 -0.121034  0.199124  0.153641   

        비유동부채비율  
14788 -1.254256  
14789 -1.290585  
14790  3.276397  
14791  2.658431  
14792  1.983518  


In [5]:
df_cleaned = create_date_column(df_cleaned)

In [6]:
df_cleaned = create_date_column(df_cleaned)

In [7]:
df_long = convert_to_long_format(df_cleaned)

# inf → NaN, 그 뒤 객체형 변환 보정
df_long = df_long.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

# 또는 특정 열만 안전하게 처리
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long = df_long.dropna(subset=["value"])

new_col = ['symbol', 'company_name', 'date', 'indicator', 'value']
df_long.columns = new_col

fs_value_df = df_long

In [8]:
# db_info = {
#     "user": 'stox7412',
#     "password": 'Apt106503!~',
#     # "host": '192.168.0.230',
#     'host': 'hystox74.synology.me',
#     "port": 3307,
#     "database": "investar"
# }
#
# upload_fs_data_to_db(df_long, db_info)

In [24]:
#### valuation 지표를 업로는 한다

# 1. 파일 로드
path = r"C:\Users\MetaM\PycharmProjects\investment\data\DataGuide_Ratio.xlsx"
raw_df = pd.read_excel(path, sheet_name='PSR', header=None)

# 2. Symbol, Name, Indicator 추출
symbol_row = 8
name_row = 9
item_name_row = 12
start_data_row = 13

symbols = raw_df.iloc[symbol_row, 2:].values
names = raw_df.iloc[name_row, 2:].values
indicators = raw_df.iloc[item_name_row, 2:].values

# 3. 날짜 데이터 추출 + to_datetime 처리 (오류 무시)
raw_dates = raw_df.iloc[start_data_row:, 0]
parsed_dates = pd.to_datetime(raw_dates, format="%Y-%m-%d", errors='coerce')

# 4. long format으로 정리
records = []
for col_idx, symbol in enumerate(symbols):
    if pd.isna(symbol):
        continue
    name = names[col_idx]
    indicator = indicators[col_idx]
    values = raw_df.iloc[start_data_row:, col_idx + 2].values
    for date, value in zip(parsed_dates, values):
        if pd.isna(date) or pd.isna(value):
            continue
        records.append({
            'date': date,
            'symbol': symbol,
            'company_name': name,
            'indicator': indicator,
            'value': value
        })

# 5. 결과 DataFrame
value_ratio_df = pd.DataFrame(records)
print(value_ratio_df.head())


# [["symbol", "company_name", "date", "indicator", "value"]]

        date   symbol company_name      indicator    value
0 2004-01-31  A000660       SK하이닉스  수정PSR(연율화)(배)  0.76583
1 2004-02-29  A000660       SK하이닉스  수정PSR(연율화)(배)  0.76754
2 2004-03-31  A000660       SK하이닉스  수정PSR(연율화)(배)  1.04821
3 2004-04-30  A000660       SK하이닉스  수정PSR(연율화)(배)  0.78614
4 2004-05-31  A000660       SK하이닉스  수정PSR(연율화)(배)  0.74320


In [29]:
concated_df = pd.concat([fs_value_df, value_ratio_df])

In [31]:
db_info = {
    "user": 'stox7412',
    "password": 'Apt106503!~',
    # "host": '192.168.0.230',
    'host': 'hystox74.synology.me',
    "port": 3307,
    "database": "investar"
}

upload_fs_data_to_db(concated_df, db_info)

✅ 총 998755개 row 업로드 완료 (중복은 자동 업데이트됨)


In [33]:
concated_df[concated_df['indicator'].str.contains('매출액')]

,symbol,company_name,date,indicator,value
0,A000080,하이트진로,2004-03-31,매출액(천원),1.595150e+08
1,A000080,하이트진로,2004-06-30,매출액(천원),1.633107e+08
2,A000080,하이트진로,2004-09-30,매출액(천원),1.866913e+08
3,A000080,하이트진로,2004-12-31,매출액(천원),1.835367e+08
4,A000080,하이트진로,2005-03-31,매출액(천원),1.785102e+08
...,...,...,...,...,...
563196,A460860,동국제강,2025-03-31,YoY_매출액,-2.175312e-01
563280,A462870,시프트업,2024-03-31,YoY_매출액,-1.675829e-01
563282,A462870,시프트업,2024-09-30,YoY_매출액,5.231445e-01
563284,A462870,시프트업,2025-03-31,YoY_매출액,1.300029e-01


In [34]:
concated_df.columns.tolist()

['symbol', 'company_name', 'date', 'indicator', 'value']

In [ ]:
dfd